In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import csv
import argparse
import datetime
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.func import functional_call, vmap, jacrev

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("high")   # overrides the matmul line above: TF32 on for fp32 matmuls (convs stay full fp32)

In [ ]:


# ==========================================================================
# 1. Model: ring-correlator embedding + 2D ResNet-CNN neural quantum state
# ==========================================================================

class RingCorrelatorEmbed(nn.Module):
    """
    First layer of the network. Produces one channel per ring radius:

      - ring 1  : the raw spin configuration, passed straight through
                  (no correlator, no multiplication).
      - ring r  : (r = 2, 4, 8, ... up to floor(L/2)), a correlator feature
                  s_i * sum_p w_{r,p} * s_{i+offset_p}
                  where the sum runs over all sites p on the square ring
                  (Chebyshev distance r) around site i, and each ring
                  position p has its own learned weight w_{r,p}.
    """

    def __init__(self, L: int):
        super().__init__()

        max_r = L // 2
        radii = []
        r = 1
        while r < max_r:
            radii.append(r)
            r *= 2
        radii.append(max_r)
        self.radii = radii  # e.g. [1, 2, 4, 8] depending on L

        # Ring 1 is a raw passthrough -> no parameters.
        # Rings > 1 each get one learned weight per ring position.
        self.ring_weights = nn.ParameterDict()
        for r in self.radii:
            if r == 1:
                continue

            offs = [
                (dx, dy)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if max(abs(dx), abs(dy)) == r
            ]

            w = torch.randn(len(offs)) * (1.0 / len(offs)) ** 0.5
            self.ring_weights[str(r)] = nn.Parameter(w)

            # flat positions of the ring inside the (2r+1)x(2r+1) kernel;
            # non-persistent buffer: follows .to(device) but is not in the state_dict
            K = 2 * r + 1
            flat_idx = torch.tensor(
                [(dx + r) * K + (dy + r) for (dx, dy) in offs],
                dtype=torch.long,
            )
            self.register_buffer(f"ring_idx_{r}", flat_idx, persistent=False)

    def _ring_kernel(self, r: int) -> torch.Tensor:
        """(1, 1, 2r+1, 2r+1) conv kernel for ring r: the learned weights on the
        ring positions, zero everywhere else."""
        K = 2 * r + 1
        w = self.ring_weights[str(r)]
        idx = getattr(self, f"ring_idx_{r}")

        k_flat = w.new_zeros(K * K).scatter(0, idx, w)
        return k_flat.view(1, 1, K, K)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        # s: (B, L, L) raw spins in {-1, +1}
        s_in = s.unsqueeze(1).float()  # (B, 1, L, L)
        channels = []
        for r in self.radii:
            if r == 1:
                channels.append(s_in)  # raw passthrough channel
                continue
            kernel = self._ring_kernel(r)
            padded = F.pad(s_in, (r, r, r, r), mode="circular")
            ring_sum = F.conv2d(padded, kernel)         # (B, 1, L, L)
            channels.append(s_in * ring_sum)             # correlator w/ center spin
        return torch.cat(channels, dim=1)  # (B, n_rings, L, L)


class ResBlock2D(nn.Module):
    """Pre-activation residual block with circular (periodic-BC) convolutions."""

    def __init__(self, channels: int = 8, kernel_size: int = 3):
        super().__init__()
        pad = kernel_size // 2
        self.norm  = nn.GroupNorm(1, channels)
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, padding=pad, padding_mode="circular")
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, padding=pad, padding_mode="circular")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.gelu(self.norm(x))
        h = F.gelu(self.conv1(h))
        h = self.conv2(h)
        return x + h


class NQS_CNN_2D(nn.Module):
    """
    2D ResNet-CNN NQS ansatz (Purely Real).
    Input : s ∈ {±1}^{L×L},  shape (batch, L, L)
    Output: ln Ψ_θ(s),       shape (batch,), float32
    """

    def __init__(self, L: int, channels: int = 8, depth: int = 4, kernel_size: int = 3):
        super().__init__()
        self.ring_embed = RingCorrelatorEmbed(L)
        n_ring_channels = len(self.ring_embed.radii)

        self.proj = nn.Conv2d(n_ring_channels, channels, kernel_size=1)

        self.blocks     = nn.ModuleList([ResBlock2D(channels, kernel_size) for _ in range(depth)])
        self.final_norm = nn.GroupNorm(1, channels)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        x = self.ring_embed(s)          # (B, n_rings, L, L)
        x = F.gelu(x)                    # activation before entering the ResNet
        x = self.proj(x)                 # (B, channels, L, L)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)

        # Flatten all features and pool them into a single real number per sample
        return torch.logsumexp(x.flatten(1), dim=1)     # (B,) real


# ==========================================================================
# 2. Monte Carlo sampler: batched Metropolis with single spin-flip proposals
# ==========================================================================

class MetropolisSampler:
    """
    n_chains independent Metropolis chains.  All chains are updated in one
    batched forward pass per step — the parallelism is across chains, not time.
    """

    def __init__(self, model: NQS_CNN_2D, L: int, n_chains: int = 128, device: str = "cpu"):
        self.model    = torch.compile(model, mode="reduce-overhead")
        self.L        = L
        self.n_chains = n_chains
        self.device   = device
        self._bn      = torch.arange(n_chains, device=device)   # cached batch index
        self.n_accepted_per_chain = torch.zeros(n_chains, dtype=torch.long, device=device)
        self.n_proposed_per_chain = 0

        self.state = torch.where(
            torch.rand(n_chains, L, L, device=device) < 0.5,
            torch.full((), -1.0, device=device),
            torch.full((), +1.0, device=device),
        )

    @torch.no_grad()
    def sweep(self, n_sweeps: int = 1) -> torch.Tensor:
        self.lnpsi = self.model(self.state).clone()
        L, n = self.L, self.n_chains
        total = n_sweeps * L * L
        bn    = self._bn

        ri = torch.randint(0, L, (total, n), device=self.device)
        rj = torch.randint(0, L, (total, n), device=self.device)
        ru = torch.rand(total, n, device=self.device)
        n_acc = torch.zeros(n, dtype=torch.long, device=self.device)   # per-chain accept count, stays on device

        for t in range(total):
          i, j = ri[t], rj[t]

          # flip unconditionally, then undo rejected flips below: every op keeps a
          # fixed shape, which the CUDA graphs behind mode="reduce-overhead" need
          self.state[bn, i, j] *= -1

          lnpsi_new = self.model(self.state).clone()
          accept = ru[t].log() < 2.0 * (lnpsi_new - self.lnpsi)
          n_acc += accept

          current = self.state[bn, i, j]
          sign = torch.where(accept, 1.0, -1.0)
          self.state[bn, i, j] = current * sign

          self.lnpsi = torch.where(accept, lnpsi_new, self.lnpsi)

        self.n_accepted_per_chain = n_acc
        self.n_proposed_per_chain = total
        return self.state

    def acceptance_stats(self) -> dict:
        """MCMC diagnostics for the most recent sweep() call."""
        acc = self.n_accepted_per_chain.double() / max(self.n_proposed_per_chain, 1)
        return {
            "n_accepted"     : int(self.n_accepted_per_chain.sum().item()),
            "n_proposed"     : self.n_proposed_per_chain * self.n_chains,
            "acc_ratio"      : acc.mean().item(),
            "acc_chain_min"  : acc.min().item(),
            "acc_chain_max"  : acc.max().item(),
            "n_frozen_chains": int((self.n_accepted_per_chain == 0).sum().item()),  # chains that never moved
        }


# ==========================================================================
# 3. Hamiltonian: vectorised local energy for 2D TFIM
# ==========================================================================

def local_energy_tfim(
    model      : NQS_CNN_2D,
    s          : torch.Tensor,
    J          ,
    h          ,
    lnpsi_s    : torch.Tensor = None,
    chunk_size : int   = 8192,
) -> torch.Tensor:
    """
    E_loc(s) = -J Σ_{<ij>} s_i s_j  -  h Σ_i [Ψ(s^i)/Ψ(s)]
    """
    B, L, _ = s.shape
    device   = s.device
    N        = L * L

    diag = -J * (
        (s * torch.roll(s, -1, 1)).sum((1, 2)) +
        (s * torch.roll(s, -1, 2)).sum((1, 2))
    )

    if lnpsi_s is None:
        lnpsi_s = model(s)

    flip_i = torch.arange(L, device=device).repeat_interleave(L)
    flip_j = torch.arange(L, device=device).repeat(L)

    s_all  = s.unsqueeze(1).expand(B, N, L, L).clone().reshape(B * N, L, L)
    bn_idx = torch.arange(B * N, device=device)
    s_all[bn_idx, flip_i.repeat(B), flip_j.repeat(B)] *= -1

    lnpsi_flip = torch.cat(
        [model(chunk) for chunk in s_all.split(chunk_size)]
    ).reshape(B, N)

    offdiag = -h * torch.exp(lnpsi_flip - lnpsi_s.unsqueeze(1)).sum(1)
    return diag.to(torch.complex64) + offdiag


# ==========================================================================
# 4. SPRING momentum — dual/kernel formulation
# ==========================================================================

def compute_lnpsi_jacobian(model: NQS_CNN_2D, s: torch.Tensor) -> torch.Tensor:
    params = {k: v.detach() for k, v in model.named_parameters()}
    param_names = list(params.keys())

    def lnpsi_fn(params_dict, s_single):
        return functional_call(model, params_dict, (s_single.unsqueeze(0),))[0]

    jac = vmap(jacrev(lnpsi_fn), in_dims=(None, 0))(params, s)

    flat_chunks = []
    for name in param_names:
        j = jac[name] # (n_chains, *param_shape)
        flat_chunks.append(j.reshape(s.shape[0], -1))

    return torch.cat(flat_chunks, dim=1) # (n_chains, n_params) float32


def sr_spring_update(
    O          : torch.Tensor,
    e_loc      : torch.Tensor,
    p_prev     : torch.Tensor,
    mu         : float = 0.9,
    diag_shift : float = 1e-3,
) -> torch.Tensor:
    n = O.shape[0]
    Obar = O - O.mean(0, keepdim=True)
    ebar = (e_loc - e_loc.mean()).to(O.dtype)

    correction = Obar @ p_prev.to(O.dtype)
    eps_tilde  = ebar - mu * correction

    # Promote Obar and eps_tilde to FP64 BEFORE matrix multiplication to avoid TF32 noise
    dtype_64 = torch.complex128 if O.is_complex() else torch.float64
    Obar_64 = Obar.to(dtype_64)
    eps_64  = eps_tilde.to(dtype_64)

    # Compute T in FP64
    T = Obar_64 @ Obar_64.conj().T

    # Explicitly enforce symmetry/Hermiticity to guarantee real non-negative eigenvalues
    T = 0.5 * (T + T.conj().T)

    # Add diagonal shift
    diag_indices = torch.arange(n, device=T.device)
    T[diag_indices, diag_indices] += diag_shift

    # Solve linear system in double precision
    y_64 = torch.linalg.solve(T, eps_64)
    x_64 = (Obar_64.conj().T @ y_64).real
    x = x_64.to(p_prev.dtype)

    return mu * p_prev + x


# ==========================================================================
# 5. Training loop: VMC with SR+SPRING momentum
# ==========================================================================

def train(
    L               ,
    depth           ,
    channels        ,
    n_chains        ,
    n_iters         ,
    burn_in_sweeps  ,
    sweeps_per_iter ,
    J               ,
    h               ,
    device          ,
    chunk_size      ,
    max_grad_norm   ,
    sr_lr           ,
    sr_diag_shift_start ,
    sr_diag_shift_end   ,
    sr_momentum     ,
    diag_interval   ,
    time_breakdown  ,
    init_model = None,
    log_rows   = None,
):
    # log_rows: list that gets one dict per completed iteration. Pass your own
    # list so the log survives even if train() itself is interrupted.
    if log_rows is None:
        log_rows = []
    if init_model is not None:
        model = init_model.to(device)   # warm start from a pre-trained model (section 7)
    else:
        model = NQS_CNN_2D(L=L, channels=channels, depth=depth).to(device)
    sampler = MetropolisSampler(model, L, n_chains=n_chains, device=device)
    sampler.sweep(burn_in_sweeps)

    n_params = sum(prm.numel() for prm in model.parameters())
    print(f"[info] optimizer = spring   n_params = {n_params}   n_chains = {n_chains}")
    print(f"[info] ring radii = {model.ring_embed.radii}   n_ring_channels = {len(model.ring_embed.radii)}")
    print(f"[info] burn-in: {burn_in_sweeps} sweeps, acceptance = {sampler.acceptance_stats()['acc_ratio']:.3f}")
    if diag_interval > 0:
        print(f"[info] full FP64 eigendecomposition diagnostics ON, every {diag_interval} iters")
    else:
        print("[info] full FP64 eigendecomposition diagnostics OFF (diag_interval=0)")

    p_prev = torch.zeros(n_params, device=device)
    energy_history = []

    timings = {"sweep": 0.0, "jacobian": 0.0, "linalg": 0.0}
    warmup_iters = 5
    is_cuda = torch.device(device).type == "cuda"

    # eigen-diagnostics, only computed on run_full_diag iterations
    lam_min = lam_max = cond_num = None
    eff_rank = None

    iters_completed = 0
    t_start = time.perf_counter()

    try:
        for it in range(n_iters):
            iters_completed = it + 1

            if time_breakdown and is_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            sampler.sweep(sweeps_per_iter)

            if time_breakdown and is_cuda:
                torch.cuda.synchronize()
            t1 = time.perf_counter()

            s = sampler.state.clone()

            with torch.no_grad():
                e_loc = local_energy_tfim(
                    model, s, J=J, h=h,
                    lnpsi_s=sampler.lnpsi.clone(),
                    chunk_size=chunk_size,
                )
                e_mean = e_loc.mean()
                e_var = e_loc.real.var().item() # Diagnostic: track variance

            frac = it / max(n_iters - 1, 1)
            diag_shift = sr_diag_shift_start + frac * (sr_diag_shift_end - sr_diag_shift_start)

            run_full_diag = diag_interval > 0 and (it % diag_interval == 0)

            with torch.no_grad():
                O = compute_lnpsi_jacobian(model, s)

                if time_breakdown and is_cuda:
                    torch.cuda.synchronize()
                t2 = time.perf_counter()

                # full FP64 eigendecomposition of T, O(n_chains^3): diagnostics only, gated
                if run_full_diag:
                    Obar = O - O.mean(0, keepdim=True)
                    dtype_64 = torch.complex128 if O.is_complex() else torch.float64
                    Obar_64 = Obar.to(dtype_64)

                    T_fp64 = Obar_64 @ Obar_64.conj().T
                    T_fp64 = 0.5 * (T_fp64 + T_fp64.conj().T)  # enforce Hermiticity

                    eigvals = torch.linalg.eigvalsh(T_fp64)
                    lam_min = eigvals.min().item()
                    lam_max = eigvals.max().item()
                    cond_num = lam_max / max(lam_min, 1e-15)
                    eff_rank = (eigvals > 1e-6).sum().item()

                param_norm = torch.nn.utils.parameters_to_vector(model.parameters()).norm().item()

                # the SPRING update itself always runs, whatever diag_interval is
                p_prev = sr_spring_update(O, e_loc, p_prev, mu=sr_momentum, diag_shift=diag_shift)

                step_norm = p_prev.norm()
                applied_step = p_prev * (max_grad_norm / step_norm) if step_norm > max_grad_norm else p_prev

                # Update in place. vector_to_parameters would give every parameter new
                # storage, moving the addresses the sampler's CUDA graphs were recorded with.
                offset = 0
                for prm in model.parameters():
                    numel = prm.numel()
                    prm.sub_(sr_lr * applied_step[offset:offset + numel].view_as(prm))
                    offset += numel

            if time_breakdown and is_cuda:
                torch.cuda.synchronize()
            t3 = time.perf_counter()

            if time_breakdown and it >= warmup_iters:
                timings["sweep"]    += (t1 - t0)
                timings["jacobian"] += (t2 - t1)
                timings["linalg"]   += (t3 - t2)

            e_site = e_mean.real.item() / (L * L)
            energy_history.append(e_site)

            mc = sampler.acceptance_stats()
            step_norm_val = step_norm.item()

            # one CSV row per iteration; eigen-diagnostics are blank on iters where they didn't run
            log_rows.append({
                "iter"        : it,
                "E_site"      : e_site,
                "E_var"       : e_var,
                "param_norm"  : param_norm,
                "eff_rank"    : eff_rank if run_full_diag else None,
                "lam_min"     : lam_min  if run_full_diag else None,
                "lam_max"     : lam_max  if run_full_diag else None,
                "cond_num"    : cond_num if run_full_diag else None,
                "diag_shift"  : diag_shift,
                "step_norm"   : step_norm_val,
                "step_clipped": step_norm_val > max_grad_norm,
                **mc,
                "t_sweep"     : t1 - t0,
                "t_jacobian"  : t2 - t1,
                "t_linalg"    : t3 - t2,
                "wall_time"   : t3 - t_start,
            })

            if it % 10 == 0:
                if run_full_diag:
                    print(f"iter {it:4d} | E/site = {e_site:.12f} | Var = {e_var:.5f} | "
                          f"Rank = {eff_rank}/{n_chains} | min_eig = {lam_min:.2e} | "
                          f"Cond = {cond_num:.1e} | |w| = {param_norm:.2f} | "
                          f"acc = {mc['acc_ratio']:.3f}")
                else:
                    print(f"iter {it:4d} | E/site = {e_site:.12f} | Var = {e_var:.5f} | "
                          f"|w| = {param_norm:.2f} | acc = {mc['acc_ratio']:.3f}  (full-diag off)")

    except KeyboardInterrupt:
        print(f"\n[info] KeyboardInterrupt caught after {iters_completed}/{n_iters} iterations — "
              f"stopping early. Returning the partially-trained model and energy history so far.")

    # This runs whether training finished normally or was interrupted above,
    # so you always get a timing summary (over whatever iterations actually ran)
    # and a usable (model, energy_history) to plot.
    if time_breakdown:
        n_timed = max(iters_completed - warmup_iters, 1)
        total = sum(timings.values())
        print("\n--- Per-iteration breakdown (excl. warmup, sync'd) ---")
        for k, v in timings.items():
            pct = (v / total * 100) if total > 0 else 0.0
            print(f"{k:>10}: {v/n_timed*1000:.1f} ms/iter  ({pct:.1f}%)")

    return model, energy_history


# ==========================================================================
# 6. Entry point — compatible with both Colab and CLI
# ==========================================================================

LOG_DIR = "logs"


def ask_yes_no(question):
    return input(f"{question} (yes/no): ").strip().lower() in ("y", "yes")


def save_log_prompt(log_rows, run_tag):
    """Ask whether to save the per-iteration training log as a CSV in LOG_DIR."""
    if len(log_rows) == 0:
        print("[info] No iterations logged — no training log to save.")
        return None

    path = os.path.join(LOG_DIR, f"{datetime.datetime.now():%Y%m%d-%H%M%S}_{run_tag}.csv")
    if not ask_yes_no(f"Save training log ({len(log_rows)} iterations) to '{path}'?"):
        print("[info] Training log not saved.")
        return None

    os.makedirs(LOG_DIR, exist_ok=True)
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(log_rows[0].keys()))
        writer.writeheader()
        writer.writerows(log_rows)
    print(f"[info] Training log saved to '{path}'")
    return path


p = argparse.ArgumentParser(description="Train a 2D CNN NQS on the TFIM.")
p.add_argument("--L",               type=int,   default=16,  help="lattice side length")
p.add_argument("--depth",           type=int,   default=8,   help="number of residual blocks")
p.add_argument("--channels",        type=int,   default=8,   help="conv channels")
p.add_argument("--n_iters",         type=int,   default=1000, help="optimizer steps")
p.add_argument("--J",               type=float, default=1.0,  help="Ising coupling")
p.add_argument("--h",               type=float, default=3.05,  help="transverse field")
p.add_argument("--chunk_size",      type=int,   default=8192 * 16, help="flip-batch chunk size (tune for VRAM)")
p.add_argument("--device",          type=str,   default="cuda" if torch.cuda.is_available() else "cpu")
p.add_argument("--sr_lr",           type=float, default=0.02, help="SPRING step size")
p.add_argument("--sr_diag_shift_start", type=float, default=1e-2)
p.add_argument("--sr_diag_shift_end",   type=float, default=1e-4)
p.add_argument("--sr_momentum",     type=float, default=0.9, help="SPRING momentum factor mu")
p.add_argument("--diag_interval",   type=int,   default=100,
               help="run full FP64 eigendecomposition diagnostics every N iters (0 = never)")
p.add_argument("--time_breakdown",  action=argparse.BooleanOptionalAction, default=True,
               help="print per-iteration timing breakdown (sweep/jacobian/linalg)")
p.add_argument("--max_grad_norm", type=float, default=1.0, help="clipping SR step")

is_colab = "google.colab" in sys.modules
args = p.parse_args([] if is_colab else None)

n_chains, sweeps_per_iter, burn_in_sweeps = 850, 8, 10
print(f"[info] n_chains={n_chains}, sweeps_per_iter={sweeps_per_iter}, burn_in_sweeps={burn_in_sweeps}")


def run(L, burn_in_sweeps, run_tag, init_model=None, n_iters=args.n_iters):
    """Train, plot E/site, and offer to save the training log. Returns (model, energy_history)."""
    log_rows = []   # filled in place by train(), so it survives an interrupt anywhere
    model, energy_history = None, []
    try:
        model, energy_history = train(
            L=L, depth=args.depth, channels=args.channels,
            n_chains=n_chains, n_iters=n_iters,
            sweeps_per_iter=sweeps_per_iter, burn_in_sweeps=burn_in_sweeps,
            J=args.J, h=args.h,
            device=args.device, chunk_size=args.chunk_size,
            max_grad_norm=args.max_grad_norm,
            sr_lr=args.sr_lr,
            sr_diag_shift_start=args.sr_diag_shift_start,
            sr_diag_shift_end=args.sr_diag_shift_end,
            sr_momentum=args.sr_momentum,
            diag_interval=args.diag_interval,
            time_breakdown=args.time_breakdown,
            init_model=init_model,
            log_rows=log_rows,
        )
    except KeyboardInterrupt:
        print("\n[info] KeyboardInterrupt caught outside the training loop (e.g. during burn-in or compilation).")

    warm_note = ", warm start" if init_model is not None else ""
    interrupted = len(log_rows) < n_iters

    if len(energy_history) > 0:
        print(f"\n[info] Final E/site (L={L}{warm_note}) = {energy_history[-1]:.6f}   "
              f"(over {len(energy_history)} recorded iterations)")

        plt.figure(figsize=(6, 4))
        plt.plot(energy_history)
        plt.xlabel("iteration")
        plt.ylabel("E / site")
        title_suffix = "  [partial / interrupted run]" if interrupted else ""
        plt.title(f"2D CNN NQS (SPRING{warm_note}) — {L}×{L} TFIM  (J={args.J}, h={args.h}){title_suffix}")
        plt.tight_layout()
        plt.show()
    else:
        print("\n[info] No iterations completed — nothing to plot.")

    save_log_prompt(log_rows, run_tag + ("_interrupted" if interrupted else ""))
    return model, energy_history


model, energy_history = run(args.L, burn_in_sweeps, run_tag=f"L{args.L}_h{args.h}")

In [ ]:
# Ask the user if they want to save the model parameters
save_model_choice = input("Do you want to save the model parameters? (yes/no): ").lower()

# Save the model parameters after the run finishes, if the user chooses to
if model is not None and save_model_choice == 'yes':
    torch.save(model.state_dict(), 'nqs_model_params.pt')
    print("\n[info] Model parameters saved to 'nqs_model_params.pt'")
elif model is not None:
    print("\n[info] Model parameters not saved as per user choice.")

KeyboardInterrupt: Interrupted by user

In [ ]:
# ==========================================================================
# 7. Warm start: initialise a larger-L model from the converged L=16 checkpoint
# ==========================================================================
# Near h_c a fresh network at large L samples badly, so start from L=16 instead.
# Everything except ring_embed and proj is L-independent and copied as-is.
# Rings are matched by radius; a ring that is new at this L keeps its random
# weights and gets a zero column in proj.weight, so it starts switched off.
# (r=8 wraps around the boundary at L=16 but not at L=20, so its copied
# weights are only a starting point.)

WARM_START_PATH = "nqs_model_params.pt"   # written by the save cell above (L=16)
L_new           = 20
burn_in_warm    = 50                      # chains still start random, but Psi is no longer flat
n_iters_warm    = args.n_iters


def load_warm_start(path, L, channels, depth, device):
    src = torch.load(path, map_location=device)
    src = {k.removeprefix("_orig_mod."): v for k, v in src.items()}   # in case it came from a compiled model

    model = NQS_CNN_2D(L=L, channels=channels, depth=depth).to(device)
    state = model.state_dict()

    ring_prefix = "ring_embed.ring_weights."
    # ring 1 is the raw passthrough and has no weights, so it never appears in src
    src_radii = [1] + sorted(int(k[len(ring_prefix):]) for k in src if k.startswith(ring_prefix))
    dst_radii = model.ring_embed.radii

    missing = [k for k in state if k not in src and not k.startswith(ring_prefix)]
    unused  = [k for k in src if k not in state and not k.startswith(ring_prefix)]
    if missing or unused:
        raise ValueError(f"checkpoint does not match depth={depth}, channels={channels}: "
                         f"missing {missing}, unexpected {unused}")

    for k, v in src.items():
        if k in state and k != "proj.weight":
            state[k] = v                     # load_state_dict below checks the shapes

    # proj.weight: (channels, n_rings, 1, 1) -- map input channels by ring radius
    proj_w = torch.zeros_like(state["proj.weight"])
    for i, r in enumerate(dst_radii):
        if r in src_radii:
            proj_w[:, i] = src["proj.weight"][:, src_radii.index(r)]
    state["proj.weight"] = proj_w

    model.load_state_dict(state)

    print(f"[warm start] loaded {path}: radii {src_radii} -> {dst_radii}")
    new_rings = [r for r in dst_radii if r not in src_radii]
    dropped   = [r for r in src_radii if r not in dst_radii]
    if new_rings:
        print(f"[warm start] new rings {new_rings}: random ring weights, zero projection")
    if dropped:
        print(f"[warm start] WARNING: rings {dropped} have no counterpart at L={L} and were dropped")
    return model


model_init = load_warm_start(WARM_START_PATH, L_new, args.channels, args.depth, args.device)

model_warm, energy_history_warm = run(
    L_new, burn_in_warm, run_tag=f"L{L_new}_h{args.h}_warm",
    init_model=model_init, n_iters=n_iters_warm,
)

ckpt_path = f"nqs_model_params_L{L_new}.pt"
if model_warm is not None and ask_yes_no(f"Save model parameters to '{ckpt_path}'?"):
    torch.save(model_warm.state_dict(), ckpt_path)
    print(f"[info] Model parameters saved to '{ckpt_path}'")
